<h1 style="text-align: center;">로봇 자율주행:<br>고수준 판단 및 저수준 제어</h1>

<br><br>

<div style="text-align: right; color: gray; font-style: italic;">
김규래&emsp;<br>
kkr.kyurae.kim@gmail.com&emsp;
</div><br>

---
---

## 0. 비상용

```bash
pkill -9 gz
pkill -9 gz_sim
pkill -9 ruby
```

---

## 1. 작업 폴더 생성

```bash
mkdir -p ~/robot_navigation/{colcon_ws/src,firmware,scripts}
```

---

## 2. Serial Port 권한 확인

```bash
groups
```

```text
>> ... dialout ... ...
```

---

## 3. Serial Port 권한 부여

```bash
sudo usermod -a -G dialout $USER
```

이후

```bash
reboot
```

#### 재부팅 후에 Serial Port 권한 재확인:
```bash
groups
```
```text
>> ... dialout ... ...
```
```bash
ls -l /dev/ttyUSB*
```
```text
>> crw-rw---- 1 root dialout ... ... /dev/ttyUSB0
```

---

## 4. 패키지 설치

```bash
sudo apt update
sudo apt install ros-jazzy-tf2-ros \
                 ros-jazzy-tf-transformations \
                 ros-jazzy-rmw-cyclonedds-cpp \
                 ros-jazzy-turtlebot3-gazebo \
                 ros-jazzy-turtlebot3-description \
                 ros-jazzy-turtlebot3-teleop \
                 ros-jazzy-ros-gz \
                 ros-jazzy-ros-gz-sim \
                 ros-jazzy-ros-gz-bridge \
                 ros-jazzy-rviz2 \
                 ros-jazzy-slam-toolbox \
                 ros-jazzy-nav2-map-server \
                 ros-jazzy-navigation2 ros-jazzy-nav2-bringup
```

---

## 5. ROS2 개발 환경 설정

```bash
echo "source /opt/ros/jazzy/setup.bash" >> ~/.bashrc
source ~/.bashrc
```

---

## 6. ROS2 통신 환경 설정

```bash
echo "export RMW_IMPLEMENTATION=rmw_cyclonedds_cpp" >> ~/.bashrc
source ~/.bashrc
```

---

## 7. ROS2 통신 환경 설정 (Domain ID 설정)

**현재 터미널 세션에 환경 변수 설정:**
```bash
export ROS_DOMAIN_ID=숫자
```

<br>

**환경 변수 영구 설정 (bashrc 등록):**
```bash
echo "export ROS_DOMAIN_ID=숫자" >> ~/.bashrc
source ~/.bashrc
```

---

## 8. colcon 환경 설정

```bash
cd ~/robot_navigation/colcon_ws
colcon build
echo "source ~/robot_navigation/colcon_ws/install/setup.bash" >> ~/.bashrc
source ~/.bashrc
```

---

## 9. colcon 패키지 생성

**실행 디렉토리:** `~/robot_navigation/colcon_ws/src`

```bash
ros2 pkg create pkg_name --build-type ament_python --dependencies rclpy std_msgs geometry_msgs sensor_msgs nav_msgs
```

---

## 10. colcon 패키지 빌드

**실행 디렉토리:** `~/robot_navigation/colcon_ws`
```bash
colcon build
```

<br>

**특정 패키지만 빌드:**

```bash
colcon build --packages-select pkg_name
```

---

## 11. 빌드 후 Source

**실행 디렉토리:** `~/robot_navigation/colcon_ws`
```bash
source install/setup.bash
```

<br>

**혹은 아무 디렉토리에서:**
```bash
source ~/robot_navigation/colcon_ws/install/setup.bash
```

<br>

**혹은 Source 하지 않고 새 터미널 열기**  
(`echo "source ~/robot_navigation/colcon_ws/install/setup.bash" >> ~/.bashrc` 명령어 등을 통해 `.bashrc`에 설정이 등록되어 있는 경우에 가능)

---

## 12. Demo Launch 파일

```python
from launch import LaunchDescription
from launch.actions import ExecuteProcess, RegisterEventHandler
from launch.event_handlers import OnProcessStart


def generate_launch_description():

    publisher = ExecuteProcess(
        cmd=[
            'gnome-terminal', '--', 'bash', '-c',
            'source ~/robot_navigation/colcon_ws/install/setup.bash; '
            'ros2 run pkg_name exec_name; '
            'exec bash'
        ],
        output='screen'
    )

    processor = ExecuteProcess(
        cmd=[
            'gnome-terminal', '--', 'bash', '-c',
            'source ~/robot_navigation/colcon_ws/install/setup.bash; '
            'ros2 run pkg_name exec_name; '
            'exec bash'
        ],
        output='screen'
    )

    subscriber = ExecuteProcess(
        cmd=[
            'gnome-terminal', '--', 'bash', '-c',
            'source ~/robot_navigation/colcon_ws/install/setup.bash; '
            'ros2 run pkg_name exec_name; '
            'exec bash'
        ],
        output='screen'
    )

    return LaunchDescription([
        publisher,

        RegisterEventHandler(
            OnProcessStart(
                target_action=publisher,
                on_start=[processor]
            )
        ),

        RegisterEventHandler(
            OnProcessStart(
                target_action=processor,
                on_start=[subscriber]
            )
        ),
    ])
```

---

## 13. 커스텀 메세지/서비스 (custom_interfaces) 패키지 생성

**실행 디렉토리:** `~/robot_navigation/colcon_ws/src`
```bash
ros2 pkg create pkg_name --build-type ament_cmake --dependencies rclpy std_msgs geometry_msgs sensor_msgs nav_msgs
```

---

## 14. CMakeLists.txt (전체 복사)

```cmake
cmake_minimum_required(VERSION 3.8)
project({PKG_NAME})

find_package(ament_cmake REQUIRED)
find_package(rosidl_default_generators REQUIRED)

rosidl_generate_interfaces(${PROJECT_NAME}
  "msg/{MSG_NAME}.msg"
  "srv/{SRV_NAME}.srv"
)

ament_package()
```

---

## 15. interfaces/package.xml (코드 추가)

```xml
  <build_depend>rosidl_default_generators</build_depend>
  <exec_depend>rosidl_default_runtime</exec_depend>
  <member_of_group>rosidl_interface_packages</member_of_group>
```

---

## 16. my_pkg/package.xml (코드 추가)

```xml
  <depend>interfaces_pkg_name</depend>
```

---

## 17. 시뮬레이션 환경 설정 (TurtleBot3 Burger)

```bash
echo "export TURTLEBOT3_MODEL=burger" >> ~/.bashrc
source ~/.bashrc
```

---

## 18. Gazebo 시뮬레이션 첫 시작

```bash
ros2 launch turtlebot3_gazebo 월드_이름.launch.py
```
---

## 19. Teleop (Teleoperation, 원격 조종)

**속도 지정:**
```bash
ros2 topic pub /cmd_vel geometry_msgs/msg/TwistStamped "{twist: {linear: {...}, angular: {...}}}"
```

<br>

**키보드 Teleop:**
```bash
ros2 run turtlebot3_teleop teleop_keyboard
```

---

## 20. RViz 실행 및 LaserScan 확인

```bash
rviz2
```

- Fixed Frame 설정
- Add - By display type: RobotModel
- Add - By display type: LaserScan
    - Topic 설정: /scan
    - QoS 설정: Reliability - Best Effort
---

## 21. Odometry 확인

```bash
ros2 topic echo /odom
```

---

## 22. TF 확인 (토픽 및 그래프)

```bash
ros2 topic echo /tf
ros2 run tf2_tools view_frames
```

---

## 23. Gazebo 토픽 확인

```bash
gz topic -l
gz topic -e -t /topic_name
```

**시뮬레이션 내부의 정답 Pose (Ground Truth) 확인:**
```bash
gz topic -e -t /world/default/pose/info
```

---

## 24. RViz 설정 불러오기

```bash
rviz2 -d PATH
```

---

## 25. 맵핑

```bash
ros2 launch slam_toolbox online_async_launch.py
```

---

## 26. 지도 저장

```bash
ros2 run nav2_map_server map_saver_cli -f $HOME/PATH_TO/MAP_NAME
```

---

## 27. 지도 해상도 변경

```bash
sudo nano /opt/ros/jazzy/share/slam_toolbox/config/mapper_params_online_async.yaml
```

**resolution 수정:**

```yaml
    debug_logging: false
    throttle_scans: 1
    transform_publish_period: 0.02 #if 0 never publishes odometry
    map_update_interval: 5.0
    resolution: 0.05   #  <----------- 해상도 변경
    restamp_tf: false
    min_laser_range: 0.0 #for rastering images
    max_laser_range: 20.0 #for rastering images
    minimum_time_interval: 0.5
    transform_timeout: 0.2
    tf_buffer_duration: 30.0
    stack_size_to_use: 40000000 #// program needs a larger stack size to serialize large maps
    enable_interactive_mode: true
```

---

## 28. SLAM Launch 파일 예시

```python
import os

from launch import LaunchDescription
from launch.actions import ExecuteProcess, IncludeLaunchDescription, SetEnvironmentVariable
from launch.launch_description_sources import PythonLaunchDescriptionSource
from launch_ros.substitutions import FindPackageShare
from launch.substitutions import PathJoinSubstitution


def generate_launch_description():

    # 1. Burger 모델
    set_model = SetEnvironmentVariable(
        name='TURTLEBOT3_MODEL',
        value='burger'
    )

    # 2. Turtlebot3 Gazebo
    gazebo = IncludeLaunchDescription(
        PythonLaunchDescriptionSource(
            PathJoinSubstitution([
                FindPackageShare('turtlebot3_gazebo'),
                'launch',
                'turtlebot3_world.launch.py'
            ])
        )
    )

    # 3. SLAM Toolbox
    slam_toolbox = IncludeLaunchDescription(
        PythonLaunchDescriptionSource(
            PathJoinSubstitution([
                FindPackageShare('slam_toolbox'),
                'launch',
                'online_async_launch.py'
            ])
        )
    )

    # 4. RViz2 실행 (config 포함)
    rviz = ExecuteProcess(
        cmd=[
            'rviz2',
            '-d',
            os.path.expanduser('~/.rviz2/설정파일이름.rviz')
        ],
        output='screen'
    )

    # 5. 로봇 주행 노드 실행
    navigate = ExecuteProcess(
        cmd=[
            'gnome-terminal', '--', 'bash', '-c',
            'source ~/robot_navigation/colcon_ws/install/setup.bash; '
            'ros2 run 패키지이름 주행파일이름; '
            'exec bash'
        ],
        output='screen'
    )

    return LaunchDescription([
        set_model,
        gazebo,
        slam_toolbox,
        rviz,
        navigate,
    ])
```

---

## 29. Turtlebot LiDAR 노이즈 수정

```bash
sudo cp /opt/ros/jazzy/share/turtlebot3_gazebo/models/turtlebot3_burger/model.sdf /opt/ros/jazzy/share/turtlebot3_gazebo/models/turtlebot3_burger/model.sdf.backup
sudo nano /opt/ros/jazzy/share/turtlebot3_gazebo/models/turtlebot3_burger/model.sdf
```

**Standard Deviation 수정:**

```xml
      <sensor name="hls_lfcd_lds" type="gpu_lidar">
        <always_on>true</always_on>
        <visualize>true</visualize>
        <pose>-0.032 0 0.171 0 0 0</pose>
        <update_rate>5</update_rate>
        <topic>scan</topic>
        <gz_frame_id>base_scan</gz_frame_id>
        <lidar>
          <scan>
            <horizontal>
              <samples>360</samples>
              <resolution>1.000000</resolution>
              <min_angle>0.000000</min_angle>
              <max_angle>6.280000</max_angle>
            </horizontal>
          </scan>
          <range>
            <min>0.120000</min>
            <max>3.5</max>
            <resolution>0.015000</resolution>
          </range>
          <noise>
            <type>gaussian</type>
            <mean>0.0</mean>
            <stddev>0.01</stddev>     <!-- 수정 -->
          </noise>
        </lidar>
      </sensor>
    </link>
```

---

## 30. IMU 추가

```bash
sudo nano /opt/ros/jazzy/share/turtlebot3_gazebo/models/turtlebot3_burger/model.sdf
```

**3줄 추가:**

```xml
    <link name="imu_link">
      <sensor name="tb3_imu" type="imu">
        <always_on>true</always_on>
        <update_rate>200</update_rate>
        <topic>imu</topic>
        
        <gz_frame_id>imu_link</gz_frame_id>                                 <!-- 추가 -->
        <plugin filename="gz-sim-imu-system" name="gz::sim::systems::Imu">  <!-- 추가 -->
        </plugin>                                                           <!-- 추가 -->
        
        <imu>
          <angular_velocity>
          ...
```

---

## 31. EKF 파라미터 설정

```bash
cp /opt/ros/jazzy/share/robot_localization/params/ekf.yaml ~/robot_navigation/colcon_ws/src/패키지_이름/config
```

**전체 복사:**

```yaml
ekf_filter_node:
  ros__parameters:
    # 1. 시스템 기본 주기 설정
    frequency: 30.0
    sensor_timeout: 0.1
    two_d_mode: true

    # 2. 좌표계(TF) 설정
    map_frame: map
    odom_frame: odom
    base_link_frame: base_footprint
    world_frame: odom
    publish_tf: true

    # 3. [입력 1] 바퀴 엔코더 오돔 설정
    odom0: /odom
    odom0_config: [true,  true,  false,  # 위치 (X, Y 사용 / Z 무시)
                   false, false, false,  # Roll, Pitch, Yaw 미사용 (IMU에게 맡김)
                   true,  true,  false,  # 선속도 (X, Y 속도 사용)
                   false, false, false,  # 각속도 (무시)
                   false, false, false]  # 선가속도 (무시)
    odom0_queue_size: 10

    # 4. [입력 2] IMU 센서 설정
    imu0: /imu
    imu0_config: [false, false, false,  # 위치 (IMU는 위치 측정 불가)
                  false, false, false,  # Roll, Pitch, Yaw (절대 각도 대신 회전 속도 사용)
                  false, false, false,  # 선속도 (측정 불가)
                  false, false, true,   # Z축 각속도 (Yaw 회전속도 감지)
                  true,  false, false]  # X축 선가속도 (멈춤/감속 감지)
    imu0_queue_size: 10
    imu0_remove_gravitational_acceleration: true # 중력 가속도 성분 제거

    # 5. 제어 명령(cmd_vel) 피드백 비활성화
    use_control: false


#   X,     Y,     Z,        [1~3]  3차원 위치 (Position)
#   Roll,  Pitch, Yaw,      [4~6]  3차원 자세 각도 (Orientation)
#   X_vel, Y_vel, Z_vel,    [7~9]  3차원 선속도 (Linear Velocity)
#   R_vel, P_vel, Y_vel,    [10~12] 3차원 각속도 (Angular Velocity)
#   X_acc, Y_acc, Z_acc     [13~15] 3차원 선가속도 (Linear Acceleration)
```

---

##  32. Gazebo Odom TF publish 차단 (중복 publish 제거)

**sdf 파일 수정:**

```bash
sudo nano /opt/ros/jazzy/share/turtlebot3_gazebo/models/turtlebot3_burger/model.sdf
```

```xml
    <plugin filename="gz-sim-diff-drive-system" name="gz::sim::systems::DiffDrive">

      <!-- wheels -->
      <left_joint>wheel_left_joint</left_joint>
      <right_joint>wheel_right_joint</right_joint>

      <!-- kinematics -->
      <wheel_separation>0.160</wheel_separation>
      <wheel_radius>0.033</wheel_radius>

      <!-- limits -->
      <max_linear_acceleration>1.0</max_linear_acceleration>

      <topic>cmd_vel</topic>

      <odom_topic>odom</odom_topic>
      <frame_id>odom</frame_id>
      <child_frame_id>base_footprint</child_frame_id>
      <odom_publisher_frequency>30</odom_publisher_frequency>

      <publish_odom_tf>false</publish_odom_tf>   <!-- 추가 -->
      <tf_topic>/tf</tf_topic>

    </plugin>
```

<br>

**yaml 파일 수정 (주석 처리):**

```bash
sudo cp /opt/ros/jazzy/share/turtlebot3_gazebo/params/turtlebot3_burger_bridge.yaml /opt/ros/jazzy/share/turtlebot3_gazebo/params/turtlebot3_burger_bridge.yaml.backup
sudo nano /opt/ros/jazzy/share/turtlebot3_gazebo/params/turtlebot3_burger_bridge.yaml
```

```yaml
# gz topic published by DiffDrive plugin
#- ros_topic_name: "tf"
#  gz_topic_name: "tf"
#  ros_type_name: "tf2_msgs/msg/TFMessage"
#  gz_type_name: "gz.msgs.Pose_V"
#  direction: GZ_TO_ROS
```

---

## 33. 센서융합 Launch 파일 추가 파트

```python
    # ───────────────────────────────────────────
    # [추가] EKF 센서 융합 노드
    from launch_ros.actions import Node
    
    ekf_config_path = PathJoinSubstitution([
        FindPackageShare('7_sensor_fusion'),
        'config',
        'ekf.yaml'
    ])

    robot_localization_node = Node(
        package='robot_localization',
        executable='ekf_node',
        name='ekf_filter_node',
        output='screen',
        parameters=[ekf_config_path],
    )
    # ───────────────────────────────────────────
```

---

## 34. LiDAR Scan Matching 강화 (보정 주기 조정)

```bash
sudo cp /opt/ros/jazzy/share/slam_toolbox/config/mapper_params_online_async.yaml /opt/ros/jazzy/share/slam_toolbox/config/mapper_params_online_async.yaml.backup
sudo nano /opt/ros/jazzy/share/slam_toolbox/config/mapper_params_online_async.yaml
```

```yaml
minimum_time_interval: 0.1     # 0.5 -> 0.1
minimum_travel_distance: 0.05  # 0.5 -> 0.05
minimum_travel_heading: 0.05   # 0.5 -> 0.05
```

---

## 35. /opt 파일 원상복구

```bash
sudo mv /opt/ros/jazzy/share/turtlebot3_gazebo/models/turtlebot3_burger/model.sdf.backup /opt/ros/jazzy/share/turtlebot3_gazebo/models/turtlebot3_burger/model.sdf
sudo mv /opt/ros/jazzy/share/turtlebot3_gazebo/params/turtlebot3_burger_bridge.yaml.backup /opt/ros/jazzy/share/turtlebot3_gazebo/params/turtlebot3_burger_bridge.yaml
sudo mv /opt/ros/jazzy/share/slam_toolbox/config/mapper_params_online_async.yaml.backup /opt/ros/jazzy/share/slam_toolbox/config/mapper_params_online_async.yaml
```

---
---

## A. Git

**git 최초 실행:**

```bash
git config --global user.email "you@example.com"
git config --global user.name "Your Name"
```
<br>

**로그인 정보 저장:**
```bash
git config --global credential.helper store
```

---

## B. ROS2 Jazzy 설치

```bash
sudo apt install software-properties-common
sudo add-apt-repository universe
```

```bash
sudo apt update
sudo apt install curl -y
```

```bash
export ROS_APT_SOURCE_VERSION=$(curl -s https://api.github.com/repos/ros-infrastructure/ros-apt-source/releases/latest | grep -F "tag_name" | awk -F'"' '{print $4}')
```

```bash
curl -L -o /tmp/ros2-apt-source.deb "https://github.com/ros-infrastructure/ros-apt-source/releases/download/${ROS_APT_SOURCE_VERSION}/ros2-apt-source_${ROS_APT_SOURCE_VERSION}.$(. /etc/os-release && echo ${UBUNTU_CODENAME:-${VERSION_CODENAME}})_all.deb"
```

```bash
sudo dpkg -i /tmp/ros2-apt-source.deb
```

```bash
sudo apt update
sudo apt upgrade
```

```bash
sudo apt install ros-dev-tools
sudo apt install ros-jazzy-desktop
```

---

## C. VS Code 설치

```bash
cd ~/Downloads/
wget -O vscode.deb "https://update.code.visualstudio.com/latest/linux-deb-arm64/stable"
sudo apt install ./vscode.deb
```

---

## D. Arduino IDE 설치 및 커스텀 명령어 생성

#### 설치:

```bash
sudo apt update
sudo apt install -y curl libfuse2t64
```

```bash
sudo mkdir -p /opt/arduino-ide
cd /opt/arduino-ide
```

```bash
sudo curl -fL \
    https://github.com/arduino/arduino-ide/releases/latest/download/arduino-ide_2.3.10_Linux_64bit.AppImage \
    -o arduino-ide.AppImage
```

```bash
cd /opt/arduino-ide
sudo chmod +x arduino-ide.AppImage
```

#### 테스트 실행:

```bash
cd /opt/arduino-ide
./arduino-ide.AppImage --no-sandbox
```

#### 명령어 생성:

```bash
nano ~/robot_navigation/scripts/arduino-ide
```

```bash
#!/bin/sh
exec /opt/arduino-ide/arduino-ide.AppImage --no-sandbox "$@"
```

```bash
chmod +x ~/robot_navigation/scripts/arduino-ide
sudo ln -s $HOME/robot_navigation/scripts/arduino-ide /usr/local/bin/arduino-ide
```

---

## E. Groot2 설치 및 커스텀 명령어 생성

#### 설치:

```bash
sudo mkdir -p /opt/groot2
cd /opt/groot2
sudo wget -O groot2.AppImage https://pub-32cef6782a9e411e82222dee82af193e.r2.dev/Groot2-v1.10.0-x86_64.AppImage
```

#### 테스트 실행:

```bash
cd /opt/groot2
./groot2.AppImage
```

#### 명령어 생성:

```bash
nano ~/robot_navigation/scripts/groot2
```

```bash
#!/bin/sh
exec /opt/groot2/groot2.AppImage "$@"
```

```bash
chmod +x ~/robot_navigation/scripts/groot2
sudo ln -s $HOME/robot_navigation/scripts/groot2 /usr/local/bin/groot2
```

---

## F. 인터넷 브라우저 삭제

```bash
sudo snap remove --purge firefox 2>/dev/null || true
sudo snap remove --purge chromium 2>/dev/null || true
sudo apt purge -y firefox firefox-esr chromium chromium-browser google-chrome-stable
flatpak uninstall --system --delete-data -y org.mozilla.firefox 2>/dev/null || true
flatpak uninstall --system --delete-data -y org.chromium.Chromium 2>/dev/null || true
flatpak uninstall --system --delete-data -y com.google.Chrome 2>/dev/null || true
flatpak uninstall --user --delete-data -y org.mozilla.firefox 2>/dev/null || true
flatpak uninstall --user --delete-data -y org.chromium.Chromium 2>/dev/null || true
flatpak uninstall --user --delete-data -y com.google.Chrome 2>/dev/null || true
```

---

## G. PCB–ESP32 GPIO 구성

```cpp
// Motor
#define FR_IN1 26
#define FR_IN2 25

#define FL_IN1 19
#define FL_IN2 27

#define RR_IN1 23
#define RR_IN2 12

#define RL_IN1 0
#define RL_IN2 15

// IMU (MPU6050)
#define IMU_SDA 21
#define IMU_SCL 22

// LiDAR
#define LIDAR_RX 16

// Potentiometer
#define POT_PIN 39

// Servo
#define SERVO_PIN 14

// Buzzer
#define BUZZER_PIN 5
```

---

## H. Linux CLI

#### $1)$ 터미널 프롬프트 (prompt) 이해

터미널을 실행하면 보이는 문자열:

```bash
user_name@host_name:~/parent_directory/current_directory$
```

* user_name: 현재 로그인한 사용자 이름
* host_name: 현재 접속한 컴퓨터(호스트) 이름
* @: 사용자와 컴퓨터를 구분하는 표시
* ~: 사용자의 home 디렉토리(폴더)
* /: 폴더와 폴더를 구분하는 표시
* parent_directory: 상위(부모) 디렉토리
* current_directory: 현재 디렉토리
* $: 일반 사용자 권한을 나타내는 표시 (#: 관리자 권한)

#### $2)$ 기본 터미널 명령어 종류

(a) `cd`: 디렉토리 이동 (change directory)

```bash
cd target_directory
```

<br>(b) `cd`: 홈 디렉토리로 이동

```bash
cd
```

<br>(c) `cd ..`: 상위 디렉토리로 이동

```bash
cd ..
```

<br>(d) `pwd`: 현재 디렉토리 확인 (print working directory)

```bash
pwd
```

<br>(e) `ls`: 현재 디렉토리 내 파일 목록 확인 (list)

```bash
ls
```

<br>(f) `ls -a`: 현재 디렉토리 내 숨김 파일 포함 모든 파일 확인 (list -all)

```bash
ls -a
```

<br>(g) `mkdir`: 현재 경로에 새로운 폴더 생성 (make directory)

```bash
mkdir folder_name
```

<br>(h) `mkdir -p`: 현재 경로에 새로운 폴더 생성하며 상위 폴더까지 생성 (make directory -parent)

```bash
mkdir -p parent_folder_name/folder_name
```

<br>(i) `touch`: 현재 경로에 새로운 파일 생성

```bash
touch file_name.txt
```

<br>(j) `cp`: 파일 복사 (copy)

```bash
cp file_path/file_name copy_path/new_file_name
```

<br>(k) `mv`: 파일 이동 (move)

```bash
mv file_path/file_name move_path/new_file_name
```

<br>(l) `rm`: 파일 삭제 (remove)

```bash
rm file_path/file_name
```

<br>(m) `rm -r`: 해당 폴더 및 모든 내부 파일 삭제 (remove -recursive)

```bash
rm -r folder_path
```

<br>(n) rmdir: 빈 폴더 삭제 (remove directory)

```bash
rmdir empty_folder
```

<br>(o) echo: 입력한 문자열이나 값을 터미널에 출력 혹은 파일에 저장

```bash
echo hi
echo "hi"
echo $USER
echo "hi" > test.txt
echo "hello" > test.txt
echo "again" >> test.txt
```

<br>(p) chmod: 파일/폴더의 권한 변경 (change mode)

```bash
chmod +x file_name
chmod -x file_name
chmod +w file_name
chmod -w file_name
chmod 444 file_name
chmod 644 file_name
```

<br>(q) nano: 터미널에서 GUI없이 사용하는 텍스트 편집기

```bash
nano file_name
```

<br>(r) ping: 특정 컴퓨터(호스트)와 네트워크 연결 상태를 확인

```bash
ping 8.8.8.8
ping <ip_address>
ping <host_name>.local
```

IP 주소 확인법:

```bash
hostname -I
```

<br>(s) ssh: 원격 컴퓨터(Jetson)에 접속

```bash
ssh <user_name>@<host_name>
ssh <user_name>@<ip_address>
```

<br>(t) sudo: 관리자(root) 권한으로 명령어 실행

```bash
sudo 명령어
sudo rm 관리자_권한_폴더/관리자_권한_파일
sudo nano 관리자_권한_폴더/관리자_권한_파일
```

가장 많이 사용되는 `sudo` 명령어:<br>
`sudo apt update`: 패키지 목록 갱신<br>
`sudo apt upgrade`: 설치된 패키지를 최신 버전으로 업데이트<br>
`sudo apt install <package_name>`: 패키지 설치

---

## I. 유용한 ROS2 Commands

```bash
ros2 run [pkg] [executable]
ros2 run [pkg] [executable] --ros-args -p [param]:=[value]

ros2 node list
ros2 node info /node

ros2 topic list
ros2 topic list -t
ros2 topic list -v
ros2 topic echo /topic
ros2 topic info /topic
ros2 topic info /topic -v
ros2 topic pub /topic [msg_type] "[data]"
ros2 topic hz
ros2 topic find [msg_type]

ros2 service list
ros2 service list -t
ros2 service type /service
ros2 service find [srv_type]
ros2 service call /service [srv_type] "[data]"

ros2 action list
ros2 action list -t
ros2 action info /action
ros2 action info /action -t
ros2 action send_goal /action [action_type] "[data]"
ros2 action send_goal /action [action_type] "[data]" --feedback

ros2 param list
ros2 param get /node [param]
ros2 param set /node [param] [value]
ros2 param describe /node [param]

ros2 interface list
ros2 interface show [interface]
ros2 interface packages

ros2 doctor
ros2 doctor --report
ros2 wtf

rqt_graph
ros2 run tf2_tools view_frames
rviz2 -d [PATH]/[CONFIG_FILE]
```

---
---

<br><br><div style="text-align: right; color: gray; font-style: italic;">
© 2026 김규래 (Kyu Rae Kim). All rights reserved.&emsp;<br><br>
This material is provided solely for the intended instructional purpose.&emsp;<br>
Redistribution, reproduction, modification, adaptation, or reuse of this material in any form without prior written permission from the copyright holder is prohibited.&emsp;
</div>